In [9]:
import csv
import os
import json
import random
import re
from dataclasses import replace
from pathlib import Path

import numpy as np

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

from openai import OpenAI

from NegoLib.Entities.Agents     import BUDGET_MERCHANT, LOCAL_SUPPLIER
from NegoLib.Entities.scenarios  import KIOSK_COLA, OFFICE_LOBBY
from NegoLib.Entities.strategies import ALL_MERCHANT_STRATEGIES, ALL_SUPPLIER_STRATEGIES
from NegoLib.Entities.utility    import merchant_utility, supplier_utility
from NegoLib.prompt_ensemble import (
    build_merchant_prompt,
    build_supplier_prompt,
    build_judge_prompt,
    Agent_call,
    Judge_call,
)

from NegoLib.exp_run import batch_simple_exp, quick_save


# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
# OPENAI_API_KEY is read from the process environment; never hardcode it here.

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
# client = OpenAI(
#     api_key=os.environ["OPENAI_API_KEY"],
# )
 
MODEL      = "gpt-4o"
merchant = replace(BUDGET_MERCHANT, max_rounds=10)
supplier = replace(LOCAL_SUPPLIER, max_rounds=10)
scenarios  = [KIOSK_COLA, KIOSK_COLA, OFFICE_LOBBY, OFFICE_LOBBY]

N_RANDOM_TRIALS = 3
N_EXPS_PER_TRIAL = 5
RNG_SEED = 24

output_dir = Path("results") / "A5"
output_dir.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def _extract_price_map_from_answer(answer: str, product_names: list[str]) -> dict[str, float]:
    prices = {}
    for name in product_names:
        pattern = rf"{re.escape(name)}\s*:\s*\$?\s*([0-9][0-9,]*(?:\.[0-9]+)?)\s*per unit"
        m = re.search(pattern, answer, flags=re.IGNORECASE)
        if m:
            prices[name] = float(m.group(1).replace(",", ""))

    # Fallback for irregular formatting.
    if len(prices) != len(product_names):
        raw = re.findall(r"\$\s*([0-9][0-9,]*(?:\.[0-9]+)?)\s*per unit", answer, flags=re.IGNORECASE)
        vals = [float(x.replace(",", "")) for x in raw]
        if len(vals) == len(product_names):
            prices = {name: vals[i] for i, name in enumerate(product_names)}
    return prices


def _mean_price(price_map: dict[str, float]) -> float | None:
    if not price_map:
        return None
    return float(sum(price_map.values()) / len(price_map))


def _build_round_rows_and_transitions(experiment_output: dict):
    round_rows = []
    for run_id, run in enumerate(experiment_output["runs"], start=1):
        product_names = [o["product_name"] for o in run["scenario"]["orders"]]
        for r in run["rounds"]:
            m_answer = r["merchant"]["answer"]
            s_answer = r["supplier"]["answer"]
            m_prices = _extract_price_map_from_answer(m_answer, product_names)
            s_prices = _extract_price_map_from_answer(s_answer, product_names)
            round_rows.append(
                {
                    "run_id": run_id,
                    "round": int(r["round"]),
                    "merchant_price_mean": _mean_price(m_prices),
                    "supplier_price_mean": _mean_price(s_prices),
                    "merchant_prices": m_prices,
                    "supplier_prices": s_prices,
                }
            )

    transitions = []
    by_run = {}
    for row in round_rows:
        by_run.setdefault(row["run_id"], []).append(row)

    for run_id, rows in by_run.items():
        rows = sorted(rows, key=lambda x: x["round"])
        for i in range(len(rows) - 1):
            cur = rows[i]
            nxt = rows[i + 1]
            m_n = cur["merchant_price_mean"]
            s_n = cur["supplier_price_mean"]
            s_np1 = nxt["supplier_price_mean"]
            if m_n is None or s_n is None or s_np1 is None:
                continue
            transitions.append(
                {
                    "run_id": run_id,
                    "round_n": cur["round"],
                    "merchant_n": m_n,
                    "supplier_n": s_n,
                    "supplier_n_plus_1": s_np1,
                }
            )

    return round_rows, transitions


def _fit_linear_all_points(transitions: list[dict]):
    x1 = np.array([t["merchant_n"] for t in transitions], dtype=float)
    x2 = np.array([t["supplier_n"] for t in transitions], dtype=float)
    y = np.array([t["supplier_n_plus_1"] for t in transitions], dtype=float)

    X = np.column_stack([np.ones_like(x1), x1, x2])
    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    y_hat = X @ beta

    ss_res = float(np.sum((y - y_hat) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r2 = float(1 - ss_res / ss_tot) if ss_tot > 0 else 1.0
    rmse = float(np.sqrt(np.mean((y - y_hat) ** 2)))

    return {
        "intercept": float(beta[0]),
        "coef_merchant_n": float(beta[1]),
        "coef_supplier_n": float(beta[2]),
        "rmse_all": rmse,
        "r2_all": r2,
        "n_samples": int(len(y)),
        "pred_all": y_hat.tolist(),
        "true_all": y.tolist(),
    }


def _analyze_and_save(experiment_output: dict, prefix: str):
    round_rows, transitions = _build_round_rows_and_transitions(experiment_output)

    # Save raw experiment output with strategy-rich filename.
    quick_save(experiment_output, save_path=output_dir / f"A5_{prefix}.json")

    if not transitions:
        print(f"[{prefix}] No transitions available; skip fit.")
        return {
            "prefix": prefix,
            "n_runs": len(experiment_output.get("runs", [])),
            "n_transitions": 0,
            "fit": None,
        }

    fit = _fit_linear_all_points(transitions)

    # transitions csv
    transitions_csv = output_dir / f"A5_{prefix}_transitions.csv"
    with open(transitions_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["run_id", "round_n", "merchant_n", "supplier_n", "supplier_n_plus_1"],
        )
        writer.writeheader()
        writer.writerows(transitions)

    # points + fitted csv
    fit_rows = []
    for i, t in enumerate(transitions):
        fit_rows.append(
            {
                "run_id": t["run_id"],
                "round_n": t["round_n"],
                "merchant_n": t["merchant_n"],
                "supplier_n": t["supplier_n"],
                "supplier_n_plus_1_true": fit["true_all"][i],
                "supplier_n_plus_1_fitted_linear": fit["pred_all"][i],
            }
        )

    points_fit_csv = output_dir / f"A5_{prefix}_points_and_fits_linear.csv"
    with open(points_fit_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(fit_rows[0].keys()))
        writer.writeheader()
        writer.writerows(fit_rows)

    # fit metrics json
    fit_json = output_dir / f"A5_{prefix}_linear_fit.json"
    quick_save(
        {
            "prefix": prefix,
            "fit": {k: v for k, v in fit.items() if k not in {"pred_all", "true_all"}},
        },
        save_path=fit_json,
    )

    # plots
    if plt is not None:
        y_true = np.array(fit["true_all"], dtype=float)
        y_pred = np.array(fit["pred_all"], dtype=float)

        # line plot sorted by true value
        order = np.argsort(y_true)
        fig, ax = plt.subplots(figsize=(10, 4.8))
        ax.plot(np.arange(len(order)), y_true[order], marker="o", linewidth=1.5, label="true")
        ax.plot(np.arange(len(order)), y_pred[order], linewidth=1.5, label="fitted_linear")
        ax.set_title(f"{prefix}: Linear Fit (All Points)")
        ax.set_xlabel("sample index (sorted by true value)")
        ax.set_ylabel("supplier_(n+1)")
        ax.legend()
        fig.tight_layout()
        fig.savefig(output_dir / f"A5_{prefix}_points_and_fits_linear.png", dpi=180, bbox_inches="tight")
        plt.close(fig)

        # scatter true vs fitted
        y_min = float(np.min(y_true))
        y_max = float(np.max(y_true))
        if y_max <= y_min:
            y_max = y_min + 1e-6

        fig2, ax2 = plt.subplots(figsize=(5.8, 5.2))
        ax2.scatter(y_true, y_pred, alpha=0.8)
        ax2.plot([y_min, y_max], [y_min, y_max], "k--", linewidth=1)
        ax2.set_title(f"{prefix}: True vs Fitted")
        ax2.set_xlabel("true supplier_(n+1)")
        ax2.set_ylabel("fitted supplier_(n+1)")
        fig2.tight_layout()
        fig2.savefig(output_dir / f"A5_{prefix}_true_vs_fitted_linear.png", dpi=180, bbox_inches="tight")
        plt.close(fig2)

    return {
        "prefix": prefix,
        "n_runs": len(experiment_output.get("runs", [])),
        "n_transitions": len(transitions),
        "fit": {k: v for k, v in fit.items() if k not in {"pred_all", "true_all"}},
    }


# ---------------------------------------------------------------------------
# 5 random strategy trials; each trial runs N_EXPS=3
# ---------------------------------------------------------------------------
rng = random.Random(RNG_SEED)
merchant_strategies = list(ALL_MERCHANT_STRATEGIES.keys())
supplier_strategies = list(ALL_SUPPLIER_STRATEGIES.keys())

trial_summaries = []

for trial_idx in range(1, N_RANDOM_TRIALS + 1):
    strategy_m = rng.choice(merchant_strategies)
    strategy_s = rng.choice(supplier_strategies)

    prefix = f"trial{trial_idx}_M-{strategy_m}_S-{strategy_s}"
    print(f"\n=== {prefix} | N_EXPS={N_EXPS_PER_TRIAL} ===")

    experiment_output = batch_simple_exp(
        client=client,
        MODEL=MODEL,
        N_EXPS=N_EXPS_PER_TRIAL,
        Scenarios=scenarios,
        Merchants=[merchant],
        Suppliers=[supplier],
        Merchant_strategies=[strategy_m],
        Supplier_strategies=[strategy_s],
        verbose=False,
    )

    summary = _analyze_and_save(experiment_output, prefix=prefix)
    trial_summaries.append(summary)


summary_output = {
    "metadata": {
        "model": MODEL,
        "n_random_trials": N_RANDOM_TRIALS,
        "n_exps_per_trial": N_EXPS_PER_TRIAL,
        "scenario": "KIOSK_COLA",
        "merchant_max_rounds": merchant.max_rounds,
        "supplier_max_rounds": supplier.max_rounds,
        "rng_seed": RNG_SEED,
    },
    "trials": trial_summaries,
}
quick_save(summary_output, save_path=output_dir / "A5_random_strategy_summary.json")
print("\nSaved summary:", output_dir / "A5_random_strategy_summary.json")









=== trial1_M-semantic_persuadable_S-patient_explorer | N_EXPS=5 ===

Saved to: results/A5/A5_trial1_M-semantic_persuadable_S-patient_explorer.json

Saved to: results/A5/A5_trial1_M-semantic_persuadable_S-patient_explorer_linear_fit.json

=== trial2_M-gradual_conceder_S-patient_explorer | N_EXPS=5 ===


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-0Or1pURFCzBqMC6FhBml7G2z on tokens per min (TPM): Limit 30000, Used 27890, Requested 2115. Please try again in 10ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}